# V2 MSc Project — Viva Master Code Walkthrough

**Local / Colab viva notebook only. Do not commit this file. Do not upload it to GitHub.**

Student: Syed Safi Ullah · A00073183 · University of Roehampton · MSc Artificial Intelligence

**Before running on Colab:** Runtime → Change runtime type → **GPU (T4)**. Then run the environment cells **in order** from the top.

## Purpose

- set up the Colab runtime and clone the final V2 repository
- explain the **final V2 implementation**
- show **where** important functionality is implemented
- explain **libraries** and why they are used
- provide **direct navigation** during the viva
- launch the **existing Streamlit artefact**

## THIS NOTEBOOK DOES NOT RERUN THE RESEARCH EXPERIMENT

It is a walkthrough of the final implementation and a live-demo launcher.

It does **not**:

- rerun the 420-case benchmark
- rerun calibration
- rerun the LLM judge
- rerun statistics or error analysis
- rebuild the knowledge base
- train a model
- modify V2 source code, frozen questions, or saved results

The `.py` files under `V2/` remain the source of truth. Code shown below is **read from those files**, not copied into a second implementation.

## Jump to a section

- [1. Clone the GitHub repository](#1-clone-the-github-repository)
- [2. Check GPU / T4](#2-check-gpu--t4)
- [3. Check CUDA / llama.cpp / model environment](#3-check-cuda--llamacpp--model-environment)
- [4. Move into V2](#4-move-into-v2-and-configure-python-path)
- [5. Viva-safe flags](#5-viva-safe-flags)
- [6. Project structure](#6-project-structure)
- [Libraries](#libraries-used-in-the-final-v2-implementation)
- [01 Dataset](#01-dataset)
- [02 Document processing](#02-document-processing)
- [03 Embeddings](#03-embeddings)
- [04 Vector database](#04-vector-database)
- [05 Retrieval](#05-retrieval)
- [06 Single-Agent](#06-single-agent)
- [07 Multi-Agent](#07-multi-agent)
- [08 Verification](#08-verification)
- [09 UQ](#09-uq)
- [10 Calibration](#10-calibration)
- [11 Prompts](#11-prompts)
- [12 Evaluation](#12-evaluation)
- [13 Statistics](#13-statistics)
- [14 Error analysis](#14-error-analysis)
- [15 Streamlit](#15-streamlit)
- [16 Live demo](#16-live-demo)
- [17 Viva questions](#17-viva-questions)



## Colab environment and repository setup

Run the next cells **from the top, in order**, before any code walkthrough cell.

This setup clones the repository (or reuses it), checks GPU / T4, and points Python at `V2/`.
It does **not** rerun the research experiment.



## 1. Clone the GitHub repository



In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Change these two lines if the GitHub URL or folder name is different.
REPO_URL = "https://github.com/syedsafiullah777/CAPSTONE-RAG-With-Uncertainty-Quantification-UQ-.git"
REPO_DIR = "/content/capstone-rag"


def _looks_like_repo(path: Path) -> bool:
    return (path / "V2" / "app" / "streamlit_app.py").is_file()


def _local_repo() -> Path | None:
    here = Path.cwd().resolve()
    for cand in [here, here.parent, here.parent.parent]:
        if _looks_like_repo(cand):
            return cand
        if (cand / "app" / "streamlit_app.py").is_file() and (cand / "src" / "rag" / "single_agent.py").is_file():
            return cand.parent
    mac = Path("/Users/syedsafiullah/Documents/CAPSTONE (RAG WITH UNCERTAINITY QUANTIFICATION)")
    if _looks_like_repo(mac):
        return mac
    return None


on_colab = Path("/content").exists()
repo = Path(REPO_DIR)

if on_colab:
    if _looks_like_repo(repo) or (repo / ".git").is_dir():
        print("Repository already cloned — skipping clone.")
        print("Nothing in the GitHub repository was modified.")
    else:
        print("Cloning:")
        print(" ", REPO_URL)
        print("into:")
        print(" ", REPO_DIR)
        repo.parent.mkdir(parents=True, exist_ok=True)
        result = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(repo)],
            capture_output=True,
            text=True,
        )
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
            print("CLONE FAILED. Check REPO_URL. This cell does not rebuild the research pipeline.")
        else:
            print("Clone complete. Repository files were not edited.")
else:
    local = _local_repo()
    if local is None:
        print("Not on Colab, and no local V2 checkout was found.")
        print("Set REPO_DIR to your project folder, or run this notebook on Colab.")
    else:
        REPO_DIR = str(local)
        repo = Path(REPO_DIR)
        print("Not on Colab — clone skipped.")
        print("Using the existing local repository (read-only for this notebook).")

os.chdir(str(repo) if repo.is_dir() else Path.cwd())

print()
print("Repository:")
print(" ", REPO_DIR)
print()
print("V2 directory:")
print(" ", os.path.join(REPO_DIR, "V2"))
print()

v2 = Path(REPO_DIR) / "V2"
needed = {
    "V2/src": v2 / "src",
    "V2/app": v2 / "app",
    "V2/data": v2 / "data",
    "V2/results": v2 / "results",
    "V2/notebooks": v2 / "notebooks",
}
print("Important folders:")
for label, path in needed.items():
    print(f"  {'OK     ' if path.is_dir() else 'MISSING'} {label}")
    if path.is_dir():
        names = sorted(p.name for p in path.iterdir() if not p.name.startswith("."))[:12]
        if names:
            print("           ", ", ".join(names))

print()
print("Top-level V2 files/folders:")
if v2.is_dir():
    for name in sorted(p.name for p in v2.iterdir() if not p.name.startswith(".")):
        print(" ", name)
else:
    print("  V2 directory was not found. Fix REPO_URL / REPO_DIR before continuing.")



## 2. Check GPU / T4



In [ ]:
# Hardware check. Does not run the benchmark.
!nvidia-smi

import shutil
import subprocess

if not shutil.which("nvidia-smi"):
    print("nvidia-smi not found on this machine.")
    print("On Colab: Runtime → Change runtime type → GPU (T4), then re-run this cell.")

print()

try:
    import torch
except Exception as exc:
    print("CUDA available: False")
    print("torch import failed:", exc)
    print()
    print("GPU CHECK")
    print("---------")
    print("CUDA available: False")
    print("GPU detected: none")
    print("Status: NOT READY (install/select a GPU runtime)")
else:
    cuda_ok = bool(torch.cuda.is_available())
    print("CUDA available:", cuda_ok)
    gpu_name = None
    if cuda_ok:
        gpu_name = torch.cuda.get_device_name(0)
        print("GPU:", gpu_name)
        print("CUDA version:", torch.version.cuda)
    print()
    print("GPU CHECK")
    print("---------")
    print("CUDA available:", cuda_ok)
    print("GPU detected:", gpu_name or "none")
    if cuda_ok and gpu_name and "tesla t4" in gpu_name.lower():
        print("Status: READY")
    elif cuda_ok:
        print("WARNING: Expected Tesla T4 was not detected.")
        print("Detected GPU:", gpu_name)
        print("Status: GPU PRESENT (not T4) — continuing, but this is not the official viva GPU")
    else:
        print("Status: NOT READY")



## 3. Check CUDA / llama.cpp / model environment



In [ ]:
import importlib.util
import os
import sys
from pathlib import Path

print("V2 VIVA ENVIRONMENT")
print("-------------------")
print("Python:")
print(" ", sys.version.replace("\n", " "))

gpu = "unavailable"
cuda = "unavailable"
try:
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
    cuda = str(torch.version.cuda) if torch.cuda.is_available() else "not available"
except Exception as exc:
    gpu = f"torch missing ({exc})"
    cuda = "not available"
print("GPU:")
print(" ", gpu)
print("CUDA:")
print(" ", cuda)

llama = "not importable"
try:
    import llama_cpp  # noqa: F401
    llama = "import OK"
except Exception as exc:
    llama = f"missing ({type(exc).__name__})"
print("llama.cpp:")
print(" ", llama)

# Locate a GGUF if it is already on disk. Do not download weights here.
qwen = "not found on disk (not downloaded by this cell)"
search_roots = []
if "REPO_DIR" in globals():
    search_roots.append(Path(REPO_DIR))
search_roots.extend(
    [
        Path.cwd(),
        Path("/content"),
        Path.home() / ".cache" / "huggingface" / "hub",
    ]
)
found = []
for root in search_roots:
    if not root.exists():
        continue
    try:
        for p in root.rglob("*Qwen*Q4_K_M*.gguf"):
            found.append(p)
            if len(found) >= 3:
                break
    except Exception:
        pass
    if found:
        break
if found:
    qwen = str(found[0])
print("Qwen3 model:")
print(" ", qwen)

chroma = "not importable"
try:
    import chromadb  # noqa: F401
    chroma = "import OK"
except Exception as exc:
    chroma = f"missing ({type(exc).__name__})"
print("ChromaDB:")
print(" ", chroma)

st = "not importable"
try:
    import streamlit  # noqa: F401
    st = f"import OK ({streamlit.__version__})"
except Exception as exc:
    st = f"missing ({type(exc).__name__})"
print("Streamlit:")
print(" ", st)

print()
print("This cell does not download models, rebuild the index, or rerun the 420-case experiment.")



## 4. Move into V2 and configure Python path



In [ ]:
import os
import sys
from pathlib import Path

if "REPO_DIR" not in globals():
    raise RuntimeError("Run the clone cell first so REPO_DIR is defined.")

V2_DIR = os.path.join(REPO_DIR, "V2")
V2_ROOT = Path(V2_DIR)

if not (V2_ROOT / "app" / "streamlit_app.py").is_file():
    raise FileNotFoundError(
        f"V2 source was not found at {V2_DIR}. "
        "Re-run the clone cell and check REPO_URL / REPO_DIR."
    )

os.chdir(V2_DIR)
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))
os.environ["PYTHONPATH"] = str(V2_ROOT) + os.pathsep + os.environ.get("PYTHONPATH", "")

print("Using V2:")
print(V2_DIR)
print()
print("cwd:", os.getcwd())
print("sys.path[0]:", sys.path[0])



## 5. Viva-safe flags

Keep these flags **False** for the viva, except `RUN_LIVE_DEMO`.

Expensive research stages are **not executed** from this notebook even if a flag is flipped.
The notebook will print the original script command instead. Frozen results are shown from disk.



In [ ]:
# === VIVA-SAFE FLAGS (default = do not rerun the experiment) ===
RUN_BENCHMARK = False
RUN_CALIBRATION = False
REBUILD_KB = False
RUN_JUDGE = False
RUN_STATISTICS = False
RUN_ERROR_ANALYSIS = False
RUN_LIVE_DEMO = True  # launch the existing Streamlit app only

print("VIVA MODE")
print("---------")
print("Benchmark: DISABLED" if not RUN_BENCHMARK else "Benchmark: ENABLED (this notebook will still refuse to run it)")
print("Calibration: DISABLED" if not RUN_CALIBRATION else "Calibration: ENABLED (this notebook will still refuse to run it)")
print("KB rebuild: DISABLED" if not REBUILD_KB else "KB rebuild: ENABLED (this notebook will still refuse to rebuild)")
print("Judge: DISABLED" if not RUN_JUDGE else "Judge: ENABLED (this notebook will still refuse to run it)")
print("Statistics: DISABLED" if not RUN_STATISTICS else "Statistics: ENABLED (this notebook will still refuse to run it)")
print("Error analysis: DISABLED" if not RUN_ERROR_ANALYSIS else "Error analysis: ENABLED (this notebook will still refuse to run it)")
print("Live demo: ENABLED" if RUN_LIVE_DEMO else "Live demo: DISABLED")



In [ ]:
from __future__ import annotations

import ast
import csv
import json
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import Code, Markdown, display

if "V2_ROOT" not in globals() or V2_ROOT is None:
    raise RuntimeError("Run the clone cell and the V2 path cell first.")

if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))
os.chdir(V2_ROOT)


def show_source(rel: str, function: str | None = None, start: int | None = None, end: int | None = None, max_lines: int = 90) -> None:
    """Read and display code from the live V2 source file."""
    path = V2_ROOT / rel
    if not path.is_file():
        display(Markdown(f"**Missing file:** `{rel}`"))
        return
    text = path.read_text(encoding="utf-8")
    lines = text.splitlines()
    label = rel
    if function:
        tree = ast.parse(text)
        found = None
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and node.name == function:
                found = node
                break
        if found is None:
            display(Markdown(f"**Function `{function}` not found in `{rel}`.**"))
            return
        start, end = found.lineno, found.end_lineno
        label = f"{rel} :: `{function}()`  (lines {start}–{end})"
    elif start is not None:
        end = end or start
        label = f"{rel}  (lines {start}–{end})"
        snippet = "\n".join(lines[start - 1 : end])
        display(Markdown(f"**Source (live file):** `{label}`"))
        display(Code(snippet, language="python" if path.suffix == ".py" else "yaml"))
        return
    if start is not None:
        snippet = "\n".join(lines[start - 1 : end])
    else:
        snippet = text if len(lines) <= max_lines else "\n".join(lines[:max_lines]) + "\n# ... truncated; open the file for the rest"
        label = f"{rel}  (first {min(len(lines), max_lines)} lines)"
    display(Markdown(f"**Source (live file):** `{label}`"))
    display(Code(snippet, language="python" if path.suffix in {".py"} else "yaml"))


def show_yaml_keys(rel: str, keys: list[str]) -> None:
    import yaml
    path = V2_ROOT / rel
    data = yaml.safe_load(path.read_text(encoding="utf-8"))
    subset = {k: data.get(k) for k in keys}
    display(Markdown(f"**Live YAML (selected keys):** `{rel}`"))
    display(Code(yaml.safe_dump(subset, sort_keys=False), language="yaml"))


def refuse_research(flag: bool, name: str, command: str) -> None:
    if not flag:
        display(Markdown(f"**VIVA SAFE:** `{name}` is OFF. Frozen artefacts will be shown. Nothing was rebuilt."))
        return
    display(Markdown(
        f"**Refused.** `{name}` is True, but this walkthrough notebook still will not execute the research pipeline.\n\n"
        f"If you ever needed it (not during viva), the original command is:\n\n`{command}`"
    ))


print("Helpers ready. Source displays will always read the current .py files.")
print("V2_ROOT =", V2_ROOT)



## 6. Project structure

## Project map

```
V2/
  data/                 frozen questions (140 TEST, 40 DEV)
  knowledge_base/       PDFs + Chroma index (already built)
  src/
    retrieval/          extract, chunk, embed, index, retrieve
    rag/                Single-Agent, Multi-Agent, verification, UQ
    calibration/        DEV-only threshold selection
    evaluation/         numeric correctness + post-hoc judge
    statistics/         McNemar / Wilcoxon / Spearman on saved results
    error_analysis/     rule-based taxonomy on saved results
    models/             llama.cpp (official), other backends
    config/             experiment.yaml + prompts.yaml loaders
  app/                  Streamlit artefact
  notebooks/            original Colab research / live-demo notebooks
  results/              frozen benchmark, judge, stats, error analysis
  config/               experiment.yaml, prompts.yaml
```

| Professor asks | Go to |
|---|---|
| Where are the questions? | [01 Dataset](#01-dataset) |
| Where are PDFs processed? | [02 Document processing](#02-document-processing) |
| Where are embeddings created? | [03 Embeddings](#03-embeddings) |
| Where is retrieval? | [05 Retrieval](#05-retrieval) |
| Where is Single-Agent? | [06 Single-Agent](#06-single-agent) |
| Where is Multi-Agent? | [07 Multi-Agent](#07-multi-agent) |
| Where is verification? | [08 Verification](#08-verification) |
| Where is UQ? | [09 UQ](#09-uq) |
| Where did 0.65 come from? | [10 Calibration](#10-calibration) |
| Where is evaluation? | [12 Evaluation](#12-evaluation) |
| Where are statistics? | [13 Statistics](#13-statistics) |
| Where is error analysis? | [14 Error analysis](#14-error-analysis) |
| Where is Streamlit? | [15 Streamlit](#15-streamlit) |
| How do I run it? | [16 Live demo](#16-live-demo) |


### Implementation layout (for the examiner)



In [ ]:
from pathlib import Path

root = Path(V2_DIR) if "V2_DIR" in globals() else Path(V2_ROOT)

def _list(rel: str) -> str:
    path = root / rel
    if not path.is_dir():
        return f"MISSING {rel}"
    names = sorted(p.name for p in path.iterdir() if p.suffix == ".py" and p.name != "__init__.py")
    return ", ".join(names) if names else "(no .py files)"

print("V2 source files available:")
print(" ", root / "src")
print(" ", _list("src"))
print()
print("RAG implementations:")
print(" ", root / "src" / "rag")
print(" ", _list("src/rag"))
print()
print("Retrieval modules:")
print(" ", root / "src" / "retrieval")
print(" ", _list("src/retrieval"))
print()
print("Evaluation modules:")
print(" ", root / "src" / "evaluation")
print(" ", _list("src/evaluation"))
print()
print("Streamlit application:")
app = root / "app" / "streamlit_app.py"
print(" ", app)
print(" ", "OK" if app.is_file() else "MISSING")



In [ ]:
print("Key files exist?")
for rel in [
    "data/final/selected_140_questions.csv",
    "data/calibration/calibration_questions.csv",
    "src/rag/single_agent.py",
    "src/rag/multi_agent.py",
    "src/rag/multi_agent_uq.py",
    "src/rag/verification.py",
    "src/rag/uncertainty.py",
    "src/retrieval/retriever.py",
    "src/calibration/select.py",
    "src/evaluation/judge.py",
    "src/statistics/tests.py",
    "src/error_analysis/taxonomy.py",
    "app/streamlit_app.py",
    "results/config/threshold.lock.json",
    "results/metrics/phase16_summary.csv",
    "notebooks/colab_phase21_final_live_demo.ipynb",
]:
    p = V2_ROOT / rel
    print(f"  {'OK' if p.is_file() else 'MISSING':7} {rel}")


## Libraries used in the final V2 implementation

The next cell **scans real imports** under `V2/src` and `V2/app`. The table after it explains only libraries that actually appear.

**LangGraph is not used in the final V2 source.** Multi-Agent is sequential Python (`run_multi_agent` → `compute_verification_result`). If asked about a graph library, say that honestly.

**Ollama** exists as an optional local-dev backend (`src/models/ollama_backend.py`). The official live demo **forbids** it and locks `llama_cpp`. Do not present Ollama as the viva stack.

**Official RAGAS is not used.** Faithfulness is a custom LLM-as-judge, labelled in code as custom / RAGAS-inspired.



In [ ]:
import ast
from collections import defaultdict

third_party = defaultdict(set)
skip_prefixes = ("src.",)
stdlib_hint = {
    "csv", "json", "os", "sys", "re", "math", "time", "hashlib", "logging",
    "pathlib", "typing", "dataclasses", "datetime", "collections", "random",
    "functools", "argparse", "ast", "platform", "subprocess", "shutil",
    "itertools", "statistics",
}

for folder in [V2_ROOT / "src", V2_ROOT / "app"]:
    for path in folder.rglob("*.py"):
        try:
            tree = ast.parse(path.read_text(encoding="utf-8"))
        except SyntaxError:
            continue
        rel = str(path.relative_to(V2_ROOT))
        for node in ast.walk(tree):
            names = []
            if isinstance(node, ast.Import):
                names = [a.name.split(".")[0] for a in node.names]
            elif isinstance(node, ast.ImportFrom) and node.module:
                names = [node.module.split(".")[0]]
            for name in names:
                if name in stdlib_hint or name.startswith("src"):
                    continue
                third_party[name].add(rel)

print("Third-party / non-stdlib modules found in V2/src and V2/app:\n")
for name in sorted(third_party):
    where = ", ".join(sorted(third_party[name])[:4])
    extra = " ..." if len(third_party[name]) > 4 else ""
    print(f"  {name:22}  {where}{extra}")


| Library / module | Where used | Purpose in this project |
|---|---|---|
| **PyYAML** (`yaml`) | `src/config/loader.py` | Loads `experiment.yaml` and `prompts.yaml`. If removed, the pipelines have no config. |
| **PyMuPDF** (`fitz`) | `src/retrieval/extract.py` | Extracts text from annual-report PDFs. If removed, the knowledge base cannot be built from PDFs. |
| **sentence-transformers** | `src/retrieval/embeddings.py` | Runs **BAAI/bge-small-en-v1.5** to turn text into vectors. |
| **ChromaDB** | `src/retrieval/index.py`, `retriever.py` | Stores chunk vectors and returns nearest neighbours (cosine). |
| **huggingface_hub** | `src/retrieval/pdf_fetch.py` | Downloads FinQA page PDFs from the dataset repo. |
| **datasets** | `src/data/profile_finqa.py`, `pdf_fetch.py` | Loads T²-RAGBench / FinQA metadata. Not needed at live-demo time once CSVs exist. |
| **llama_cpp** | `src/models/llama_cpp_backend.py` | Runs **Qwen3-8B Q4_K_M** on GPU (official path). |
| **NumPy** | `src/statistics/` | Arrays for paired tests. |
| **SciPy** | `src/statistics/tests.py` | McNemar (binomial), Wilcoxon, Spearman, Mann–Whitney. |
| **matplotlib** | `src/statistics/figures.py` | Saved result figures. Not required to explain code. |
| **Streamlit** | `app/streamlit_app.py` | The viva artefact UI. |
| **torch / transformers** | optional backends | Alternative generator path. Official frozen run used **llama.cpp**. |
| **ollama** | `src/models/ollama_backend.py` | Local-dev only. **Not** the official viva backend. |

Python standard library used heavily: `csv`, `json`, `pathlib`, `hashlib`, `re`, `math`, `random`, `time`.



## 01 Dataset

**Files**

- `V2/data/final/selected_140_questions.csv` — frozen TEST set
- `V2/data/calibration/calibration_questions.csv` — DEV set used only to lock T
- `V2/src/data/select_140.py` — how the 140 were sampled (seed 42)
- `V2/src/run/subset.py` — `load_frozen_question_rows()`
- `V2/src/calibration/data.py` — loads the 40 DEV questions

**Purpose.** FinQA (inside T²-RAGBench) supplies financial questions whose answers are numbers from annual reports. TEST and DEV are separate so the abstention threshold is not tuned on the final test questions.

**Viva explanation.** “I froze 140 test questions. I used a different 40 development questions only to choose T = 0.65. I never tuned that threshold on the 140.”



In [ ]:
show_source("src/run/subset.py", function="load_frozen_question_rows")
show_source("src/data/select_140.py", function="stratified_sample")

from src.run.subset import load_frozen_question_rows
from src.calibration.data import load_calibration_questions, CALIBRATION_N

test_rows = load_frozen_question_rows()
dev_rows = load_calibration_questions()
print(f"TEST rows loaded: {len(test_rows)}  (expected 140)")
print(f"DEV  rows loaded: {len(dev_rows)}  (expected {CALIBRATION_N})")
print("First TEST ids:", [r['id'] for r in test_rows[:5]])
print("First DEV ids: ", [r['id'] for r in dev_rows[:5]])
print("Overlap of IDs:", sorted(set(r['id'] for r in test_rows) & set(r['id'] for r in dev_rows)))


### What this code does
These functions read the frozen CSV files. They do not download FinQA again and they do not sample a new 140.

### Why it is needed
The viva and the benchmark must use the same questions every time.

### What goes in
Path to `selected_140_questions.csv` or the DEV CSV.

### What comes out
A list of question dicts: id, question text, program_answer, file name, company.

### Libraries used here
`csv`, `pathlib`; sampling historically used `random.Random(42)` in `select_140.py`.

### Viva questions
- **Where did the questions come from?**
  - FinQA subset of T²-RAGBench (`G4KMU/t2-ragbench`), configured in `config/experiment.yaml`.
- **Why FinQA?**
  - Financial annual-report QA matches the enterprise document setting and needs exact numbers.
- **Why 140?**
  - A fixed, paired test size. `frozen_test_size: 140` in `experiment.yaml`; sampled with seed 42.
- **Why a separate DEV set?**
  - So T = 0.65 is chosen without looking at TEST. `used_frozen_test_140` is false in the lock file.
- **Did you tune the threshold on TEST?**
  - No. Lock file: `source_split: dev`, `used_frozen_test_140: false`.



In [ ]:
lock = json.loads((V2_ROOT / "results/config/threshold.lock.json").read_text())
print("locked T           =", lock["threshold"])
print("source_split       =", lock["source_split"])
print("used_frozen_test_140 =", lock["used_frozen_test_140"])
print("n DEV cases        =", lock["n"])
print("rule               =", lock["rule"])


## 02 Document processing

**Pipeline (already built; do not rebuild in viva):**

230 annual-report PDFs → extract text → chunk (900 / 150) → embed (BGE-small) → ChromaDB (`finqa_source_pdfs`)

| Setting | Value | Where |
|---|---|---|
| PDFs | 230 | `knowledge_base/index/index_manifest.json` |
| Chunks | 1,239 | same manifest |
| Chunk size | 900 characters | `chunking.py`, `experiment.yaml` |
| Overlap | 150 characters | same |
| Embedding | `BAAI/bge-small-en-v1.5` | `embeddings.py` |
| Store | Chroma, cosine | `index.py` (`hnsw:space: cosine`) |
| top-k | 4 | `retriever.py`, `experiment.yaml` |

Gold FinQA answers are **not** ingested into the searchable index.



In [ ]:
show_source("src/retrieval/extract.py", function="extract_pdf_pages")
show_source("src/retrieval/chunking.py", function="split_text")
manifest = json.loads((V2_ROOT / "knowledge_base/index/index_manifest.json").read_text())
print("Indexed docs:", manifest.get("docs_indexed"), " chunks:", manifest.get("chunks"))
print("Embedding   :", manifest.get("embedding_model"))
print("chunk_size  :", manifest.get("chunk_size"), "overlap:", manifest.get("chunk_overlap"))
print("roles       :", manifest.get("roles"))
refuse_research(REBUILD_KB, "REBUILD_KB", "PYTHONPATH=. python scripts/build_index.py")


### What this code does
`extract_pdf_pages` opens each PDF with PyMuPDF and reads page text. `split_text` cuts that text into overlapping character windows.

### Why it is needed
The LLM cannot search a whole annual report. Chunks are the searchable evidence units.

### What goes in
A PDF path; then page text plus chunk_size=900 and overlap=150.

### What comes out
A list of text chunks with metadata (file, company, year, page).

### Libraries used here
PyMuPDF (`fitz`); chunking itself is plain Python.

### Viva questions
- **Why 900 characters?**
  - Configured in `experiment.yaml` / `split_text` default. It is the experimental setting, not claimed as globally optimal.
- **Why overlap?**
  - So a sentence or table row is less likely to be split in half. Default overlap is 150.
- **How do documents become evidence?**
  - PDF text → chunks → vectors → Chroma. At query time the question vector retrieves the nearest 4 chunks.



## 03 Embeddings



In [ ]:
show_source("src/retrieval/embeddings.py", function="embed_texts")


### What this code does
`embed_texts` loads BGE-small once, encodes a batch of strings, and L2-normalises the vectors.

### Why it is needed
Retrieval is nearest-neighbour search in that vector space.

### What goes in
A list of texts (question or chunks).

### What comes out
A list of float vectors.

### Libraries used here
`sentence_transformers.SentenceTransformer`; `functools.lru_cache` to reuse the model.

### Viva questions
- **Why BGE-small?**
  - It is the frozen experimental encoder in `experiment.yaml`. Small enough to run locally; not claimed as the best possible embedder.
- **What if you removed it?**
  - No vectors, so Chroma cannot rank chunks.



## 04 Vector database



In [ ]:
show_source("src/retrieval/index.py", function="load_collection")
show_source("src/retrieval/index.py", function="build_knowledge_base")


### What this code does
`load_collection` opens the persistent Chroma folder and the `finqa_source_pdfs` collection with cosine space. `build_knowledge_base` is how that index was created; viva mode must not call it.

### Why it is needed
This is the shared memory all three architectures query.

### What goes in
persist_dir = `knowledge_base/index`.

### What comes out
A Chroma collection object.

### Libraries used here
`chromadb.PersistentClient`.

### Viva questions
- **Why ChromaDB?**
  - Local persistent vector store. Fits a controlled research prototype. Not a claim that it is production-scale.
- **Why cosine?**
  - `metadata={"hnsw:space": "cosine"}` in `load_collection`. Retriever converts distance to similarity as `1 - distance`.



## 05 Retrieval

**File:** `V2/src/retrieval/retriever.py`  
**Function:** `retrieve()`

Question → embed question → Chroma query → top 4 chunks → evidence for every architecture.



In [ ]:
show_source("src/retrieval/retriever.py", function="retrieve")
show_source("src/retrieval/retriever.py", function="_distance_to_similarity")


### What this code does
It embeds the question with the same BGE model, asks Chroma for `n_results=top_k` (4), and converts cosine distance into a similarity-like score.

### Why it is needed
This is the shared retrieval foundation. Architectures must not use different evidence for the same question.

### What goes in
The question string plus persist_dir / top_k / embedding model / collection name.

### What comes out
A list of `RetrievedChunk` objects (text, score, file, company, year).

### Libraries used here
`embed_texts`, Chroma `collection.query`.

### Viva questions
- **What is top-k?**
  - How many nearest chunks are returned. Here k = 4.
- **What does the similarity score mean?**
  - Chroma cosine distance converted by `1 - distance`, clipped at 0. It is a ranking signal, not a probability.
- **Why is retrieval the same for all three?**
  - So differences later are due to generate / check / abstain, not due to different documents.
- **Why controlled?**
  - Internal validity: one embedding, one store, one k.



## 06 Single-Agent

**File:** `V2/src/rag/single_agent.py`  
**Function:** `run_single_agent()`

Flow: retrieve evidence → build prompt → Qwen3-8B generates an answer → always `decision="ANSWER"`.

No verification. No confidence. No abstention. That is why it is the baseline.



In [ ]:
show_source("src/rag/single_agent.py", function="run_single_agent")


### What this code does
One function runs retrieval, builds the baseline prompt, calls the LLM, cleans the text, and stores an ANSWER result.

### Why it is needed
It is the simplest RAG path. Multi-Agent and UQ are compared against this.

### What goes in
Question, optional question_id, shared config/backend.

### What comes out
`RAGCaseResult` with answer, evidence, scores, `confidence=None`, `decision="ANSWER"`.

### Libraries used here
Project modules plus `time`. The LLM is reached through `create_backend` (officially llama.cpp).

### Viva questions
- **Why is this the baseline?**
  - Retrieve then generate only. No extra checking and no option to stay silent.
- **What is the LLM’s role?**
  - Write a short answer from the retrieved chunks only.
- **What is in the prompt?**
  - Evidence + question + instruction to distinguish final value vs change vs ROI. See Prompts.
- **Why does it always answer?**
  - The result hard-codes `decision="ANSWER"`. There is no threshold.
- **Why no confidence?**
  - `confidence=None` and `threshold=None` in the returned record.



## 07 Multi-Agent

**File:** `V2/src/rag/multi_agent.py`  
**Function:** `run_multi_agent()`

Flow: retrieve → generate an answer → **check evidence** → still return that **same answer**.

The checker does **not** rewrite or correct the answer. `decision` is still `"ANSWER"`.



In [ ]:
show_source("src/rag/multi_agent.py", function="run_multi_agent")


### What this code does
After generating the answer, it calls `compute_verification_result` on that same text. The displayed answer is still the generated answer.

### Why it is needed
This isolates the effect of checking evidence without mixing in answer-repair.

### What goes in
Same inputs as Single-Agent.

### What comes out
`RAGCaseResult` with `verification_result` filled, still `decision="ANSWER"`.

### Libraries used here
Same stack as Single-Agent, plus `verification.py`. No LangGraph import.

### Viva questions
- **Why a separate verification stage?**
  - To score support after generation, without changing retrieval.
- **Does verification correct the answer?**
  - No. The generated text is returned unchanged.
- **Why can a verified answer still be wrong?**
  - VERIFIED means support score ≥ 0.50, not numeric match to FinQA gold.
- **Did you use LangGraph?**
  - Not in the final V2 source. Control flow is ordinary Python function calls.



## 08 Verification

**File:** `V2/src/rag/verification.py`  
**Function:** `compute_verification_result()`

Score = mean of (token overlap with evidence, LLM support score 0–1).  
≥ 0.50 → `VERIFIED`, else `WEAK_EVIDENCE`. Temperature 0.0, max 32 tokens.



In [ ]:
show_source("src/rag/verification.py", function="compute_verification_result")


### What this code does
It measures lexical overlap between the answer and the evidence, asks Qwen for a 0–1 support number, and averages them. It does not edit the answer.

### Why it is needed
Multi-Agent and UQ both need an evidence-support signal.

### What goes in
Question, generated answer, retrieved chunks, LLM.

### What comes out
A dict: verification_score, lexical_score, llm_score, status, rationale.

### Libraries used here
Plain Python helpers in `text_utils.py`; LLM via `LLMBackend.generate`.

### Viva questions
- **What is the verifier checking?**
  - Whether the answer is supported by retrieved text and whether it answers the asked quantity.
- **How is the score produced?**
  - Average of token-overlap and a parsed LLM 0.00–1.00 score.
- **Why temperature 0.0?**
  - The call uses `temperature=0.0` so the support number is more stable.



## 09 UQ

**Files:** `V2/src/rag/multi_agent_uq.py`, `V2/src/rag/uncertainty.py`

**Actual formula in code:**

`retrieval_score` = mean of the 4 chunk similarities  
`verification_score` = from the checker  
`confidence` = mean of those two  
If `confidence >= T` → ANSWER, else ABSTAIN

**T = 0.65** from the DEV lock file. Confidence is an **operational decision score**, not a calibrated probability.



In [ ]:
show_source("src/rag/uncertainty.py", function="compute_retrieval_score")
show_source("src/rag/uncertainty.py", function="compute_combined_confidence")
show_source("src/rag/uncertainty.py", function="apply_abstention_decision")
show_source("src/rag/multi_agent_uq.py", function="run_multi_agent_uq")


### What this code does
`run_multi_agent_uq` does the Multi-Agent path, then averages retrieval and verification into confidence, then keeps or replaces the answer.

### Why it is needed
This is the only architecture that can refuse to answer.

### What goes in
Same question/evidence, plus locked threshold 0.65.

### What comes out
ANSWER with the generated text, or ABSTAIN with the abstention message.

### Libraries used here
Same RAG modules; `average()` in `text_utils.py`.

### Viva questions
- **How is confidence calculated?**
  - Mean of mean retrieval similarity and verification score (`mean_retrieval_verification`).
- **Is confidence a probability?**
  - No. The code treats it as a decision score. Dissertation: not a calibrated probability.
- **What happens below 0.65?**
  - `apply_abstention_decision` returns the abstention message and `ABSTAIN`.
- **Why can UQ abstain when Single-Agent cannot?**
  - Only UQ calls `apply_abstention_decision`. Single-Agent always sets ANSWER.



## 10 Calibration

**Files:** `V2/src/calibration/select.py`, `V2/results/config/threshold.lock.json`

Rule in code: maximise selective accuracy among thresholds with **coverage ≥ 0.50**; ties take the **lowest** T.

That rule on 40 DEV questions selected **0.65** (22 ANSWER / 18 ABSTAIN on DEV). Then the value was locked before TEST.



In [ ]:
show_source("src/calibration/select.py", function="select_threshold")
show_source("src/calibration/select.py", function="metrics_at_threshold")
refuse_research(RUN_CALIBRATION, "RUN_CALIBRATION", "PYTHONPATH=. python scripts/run_calibration.py")
lock = json.loads((V2_ROOT / "results/config/threshold.lock.json").read_text())
print({k: lock[k] for k in ["threshold", "locked", "n", "n_answer", "n_abstain", "coverage", "selective_accuracy", "source_split", "used_frozen_test_140"]})


### What this code does
`select_threshold` sweeps possible T values on DEV cases only and picks the locked operating point.

### Why it is needed
Without a held-out DEV set, T could be fitted to TEST and the TEST results would be over-optimistic.

### What goes in
40 DEV UQ cases with confidence and numeric correctness vs gold.

### What comes out
The lock JSON: T=0.65, coverage 0.55, selective accuracy ≈ 0.545 on DEV.

### Libraries used here
Pure Python + `numeric_match`. No TEST CSV is read here.

### Viva questions
- **Why 0.65?**
  - It won the DEV sweep under coverage ≥ 0.50 and lowest-T tie-break.
- **Was 0.65 chosen using TEST?**
  - No. `used_frozen_test_140: false`.
- **Is 0.65 universal?**
  - No. It is the lock for this experiment.



## 11 Prompts

**Files:** `V2/src/rag/prompts.py`, `V2/config/prompts.yaml`

Generation asks for a **short** answer from evidence only, and tells the model that **final / cumulative value is not the same as absolute change, percentage change, or ROI**.

Verification asks for **one number 0.00–1.00**. Generation temperature is 0.1 (512 tokens). Verification temperature is 0.0 (32 tokens).



In [ ]:
show_source("src/rag/prompts.py", function="build_baseline_prompt")
show_source("src/rag/prompts.py", start=11, end=64)


### What this code does
They assemble system + user text with the retrieved chunks. They do not compute answers themselves.

### Why it is needed
Financial questions fail if the model reports the wrong quantity (for example ending value instead of ROI).

### What goes in
Question + retrieved chunks (+ draft answer for verification).

### What comes out
One prompt string sent to Qwen.

### Libraries used here
Plain Python; YAML via `load_prompts_config()`.

### Viva questions
- **What does the generation prompt tell the model?**
  - Use only evidence; answer once; distinguish final value vs change vs ROI; say “Evidence is insufficient” if needed.
- **Why list ROI vs ending value?**
  - Those quantities look similar in a table but FinQA gold is one specific number.
- **Why is verification temperature different?**
  - `compute_verification_result(..., temperature=0.0, max_new_tokens=32)` vs generation 0.1 / 512.



## 12 Evaluation

Two **already finished** layers:

1. **Numeric correctness (CPU)** — `src/evaluation/numeric.py` `numeric_match()` against FinQA `program_answer`. Used for the 22.86% / 20.71% figures. Script conceptually: `src/evaluation/runner.py` over saved Phase 15 cases.
2. **Post-hoc LLM-as-judge** — `src/evaluation/judge.py`. Sees **question + retrieved evidence + claim**. Does **not** see gold answers or gold context. Does **not** rerun RAG. Labelled **custom / RAGAS-inspired, not official RAGAS**.

Saved results (read only):

- `V2/results/processed/phase16_cases.jsonl`
- `V2/results/metrics/phase16_summary.csv`



In [ ]:
show_source("src/evaluation/numeric.py", function="numeric_match")
show_source("src/evaluation/metrics.py", function="score_case")
show_source("src/evaluation/judge.py", function="judge_one_case")
refuse_research(RUN_BENCHMARK, "RUN_BENCHMARK", "PYTHONPATH=. python scripts/run_benchmark.py")
refuse_research(RUN_JUDGE, "RUN_JUDGE", "PYTHONPATH=. python scripts/run_judge.py")

import csv
with (V2_ROOT / "results/metrics/phase16_summary.csv").open(encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print("Frozen Phase 16 summary (not recomputed):")
for row in rows:
    print(row["architecture"], "correctness", round(float(row["answer_correctness"]), 4),
          "coverage", round(float(row["coverage"]), 4),
          "selective", round(float(row["selective_accuracy"]), 4),
          "unsupported", round(float(row["unsupported_emitted_rate"]), 4))


### What this code does
`numeric_match` checks whether any number in the system text is close to the gold FinQA number. The judge later scores evidence support of the saved claim only.

### Why it is needed
Correctness and evidence-support are different questions. RAG can retrieve the right file and still emit the wrong number.

### What goes in
Saved RAG cases + gold `program_answer` (correctness). Judge: question, chunks, claim.

### What comes out
Per-case flags and the summary CSV already on disk.

### Libraries used here
`math`/`re` for numbers; judge uses the same Qwen backend **after** the benchmark, not during it.

### Viva questions
- **How did you evaluate correctness?**
  - Numeric match to FinQA `program_answer` with relative/absolute tolerance.
- **How did you evaluate evidence support?**
  - Custom LLM-as-judge faithfulness on saved cases; also token-overlap as a secondary CPU metric.
- **Why an LLM judge?**
  - To score support without official RAGAS and without putting gold answers in the judge prompt.
- **Limitations?**
  - Same Qwen3-8B family as generator/verifier — possible same-model bias. Not a human gold standard.
- **Why not official RAGAS?**
  - The code and dissertation say custom / RAGAS-inspired, not the official RAGAS library pipeline.



## 13 Statistics

**Files:** `V2/src/statistics/tests.py`, `V2/src/statistics/analysis.py`, `V2/scripts/run_statistics.py`

**Why paired McNemar?** The same 140 questions are run on each architecture. McNemar tests whether discordant pairs (correct on one, wrong on the other) are balanced.

**Saved:** `V2/results/metrics/phase17_tests.csv`  
RQ1 SA vs MA displayed correctness: **p = 0.6776** (not significant).



In [ ]:
show_source("src/statistics/tests.py", function="mcnemar_exact")
refuse_research(RUN_STATISTICS, "RUN_STATISTICS", "PYTHONPATH=. python scripts/run_statistics.py")

import csv
with (V2_ROOT / "results/metrics/phase17_tests.csv").open(encoding="utf-8") as f:
    tests = list(csv.DictReader(f))
print("Confirmatory tests already saved:")
for row in tests:
    if row.get("role") != "confirmatory":
        continue
    print(f"{row['id']:55} p={row['p_value']}  holm={row['p_value_holm']}  sig={row['significant_holm_0.05']}")


### What this code does
`mcnemar_exact` counts paired yes/no outcomes on the same 140 questions. Other helpers run Wilcoxon, Spearman, Mann–Whitney, Wilson intervals, Holm adjustment, and effect sizes.

### Why it is needed
A 2-point accuracy difference is not a finding by itself. The paired test answers whether that difference is distinguishable from chance.

### What goes in
Two length-140 binary series (or continuous scores) from frozen CSVs.

### What comes out
p-value, Holm-adjusted p, effect size, already written to `phase17_tests.csv`.

### Libraries used here
NumPy and SciPy (`stats.binomtest`, `wilcoxon`, `spearmanr`, `mannwhitneyu`). Statistics code is forbidden from importing the RAG generators.

### Viva questions
- **Why McNemar?**
  - Paired binary correctness on identical questions.
- **Why the same 140?**
  - Otherwise architecture is confounded with question difficulty.
- **What does p = 0.6776 mean?**
  - No statistically significant paired difference in displayed correctness between Single-Agent and Multi-Agent at α = 0.05.
- **Why effect sizes?**
  - A p-value does not describe the size of a difference; Cohen’s g / dz / rank-biserial are stored in the CSV.



## 14 Error analysis

**Files:** `V2/src/error_analysis/taxonomy.py`, `V2/scripts/run_error_analysis.py`  
**Saved:** `V2/results/analysis/phase18_error_summary.csv`

Purpose: label **why** a case failed (retrieval vs numerical reasoning vs unsupported claim vs abstention), using **saved** fields only. No new generation.



In [ ]:
show_source("src/error_analysis/taxonomy.py", function="assign_category")
refuse_research(RUN_ERROR_ANALYSIS, "RUN_ERROR_ANALYSIS", "PYTHONPATH=. python scripts/run_error_analysis.py")

import csv
with (V2_ROOT / "results/analysis/phase18_error_summary.csv").open(encoding="utf-8") as f:
    rows = [r for r in csv.DictReader(f) if r.get("scope") == "full_420_rule_based"]
print("Rule-based taxonomy counts (frozen):")
for r in rows:
    print(f"{r['architecture_label']:18} {r['primary_category']:34} n={r['n']}")


### What this code does
`assign_category` is a deterministic if/elif over stored flags (answered, numeric correct, context recall, faithfulness). It does not invent a cause from the raw PDF.

### Why it is needed
Overall accuracy hides whether the miss was “never retrieved the number” or “had the number and calculated badly”.

### What goes in
One joined saved case.

### What comes out
A primary category such as `retrieval_failure` or `incorrect_numerical_reasoning`.

### Libraries used here
Uses `parse_numbers` from evaluation; no LLM call.

### Viva questions
- **Why error analysis?**
  - To separate retrieval failure from numerical reasoning and from abstention behaviour.
- **What types of errors?**
  - On the 420-case table: correct answers, retrieval failures, non-numeric answers, incorrect numerical reasoning, unsupported claims, and (UQ) appropriate/incorrect abstention.
- **How did retrieval differ from reasoning?**
  - If the gold number is absent from the top-4 text, that is retrieval; if it is present and the displayed number is still wrong, that is labelled numerical reasoning.



## 15 Streamlit

**File:** `V2/app/streamlit_app.py`  
**Live runner:** `V2/src/rag/live.py` (`run_live_comparison`)  
**Official Colab launcher (GPU):** `V2/notebooks/colab_phase21_final_live_demo.ipynb`

Pages:

1. **Live RAG Demo** — runs the real three pipelines on a question
2. **Benchmark Results** — read-only frozen metrics
3. **Benchmark Questions** — read-only frozen 140

The app shows evidence, generated answer, verification, confidence, ANSWER/ABSTAIN, and runtime. It does **not** look up the 420 saved answers for a live question.



In [ ]:
show_source("app/streamlit_app.py", function="main")
show_source("app/streamlit_app.py", function="render_live_rag_demo")


### What this code does
`main` chooses the page. Live Demo calls `run_live_comparison`, which independently calls `run_single_agent`, `run_multi_agent`, and `run_multi_agent_uq` on the same question.

### Why it is needed
Examiners can see the mechanisms, not only a results table.

### What goes in
A typed question or a frozen catalogue id (gold is not fed into the live pipelines).

### What comes out
Three on-screen results plus a summary table.

### Libraries used here
Streamlit widgets; RAG modules above. Official GPU path is llama.cpp.

### Viva questions
- **Is this a mock app?**
  - No. Live Demo executes the real functions. Mock is only a forbidden/local-test backend.
- **Will Live Demo work on this Mac?**
  - The UI will start. Official Qwen/T4 generation is the Colab Phase 21 path. Benchmark Results still show frozen metrics without a GPU.



## 16 Live demo

This section only:

- checks that the V2 files, Chroma index, and model runtime exist
- launches the **existing** `app/streamlit_app.py`
- does **not** rebuild the index, regenerate embeddings, recalibrate T = 0.65, or rerun 420 cases

On **Colab**, open the **proxy URL** printed below (not `127.0.0.1` on your Mac).

If the index or GGUF is missing, this cell **prints what is missing**. It will not silently rebuild the research pipeline.



In [ ]:
def viva_preflight() -> dict:
    info = {"v2_root": str(V2_ROOT), "ok": True, "notes": [], "missing": []}
    required = [
        "app/streamlit_app.py",
        "src/rag/live.py",
        "data/final/selected_140_questions.csv",
        "data/calibration/calibration_questions.csv",
        "results/config/threshold.lock.json",
        "knowledge_base/index/index_manifest.json",
        "config/experiment.yaml",
        "config/prompts.yaml",
    ]
    print("Required files")
    print("--------------")
    for rel in required:
        ok = (V2_ROOT / rel).is_file()
        print(("OK     " if ok else "MISSING"), rel)
        if not ok:
            info["ok"] = False
            info["missing"].append(rel)

    manifest = V2_ROOT / "knowledge_base/index/index_manifest.json"
    if manifest.is_file():
        m = json.loads(manifest.read_text())
        print("Index chunks:", m.get("chunks"), "docs:", m.get("docs_indexed"))
        if int(m.get("chunks") or 0) != 1239:
            info["notes"].append("Chunk count is not 1239; check this checkout.")
    try:
        from src.retrieval.preflight import validate_index_preflight
        from src.config import get_path, load_experiment_config
        cfg = load_experiment_config()
        pf = validate_index_preflight(get_path(cfg, "kb_index"))
        print("Chroma preflight:", {k: pf.get(k) for k in list(pf)[:8]})
    except Exception as exc:
        info["ok"] = False
        info["missing"].append("Chroma index (preflight failed)")
        info["notes"].append(
            f"Index preflight: {exc}. Restore the existing Phase 6 knowledge base. "
            "Do not run scripts/build_index.py from this notebook."
        )
        print("Index preflight failed:", exc)
        print("Restore the existing Chroma index. This notebook will not rebuild it.")

    lock = json.loads((V2_ROOT / "results/config/threshold.lock.json").read_text())
    print("Locked T:", lock["threshold"], "DEV only:", lock["used_frozen_test_140"] is False)

    try:
        import llama_cpp  # noqa: F401
        print("llama_cpp: import OK")
    except Exception as exc:
        info["notes"].append(f"llama_cpp missing: {exc}")
        print("llama_cpp: missing — live Qwen generation will not run until it is installed.")
        print("This notebook will not download a new model automatically.")

    return info

pre = viva_preflight()
print()
print("preflight ok =", pre["ok"])
if pre["missing"]:
    print("Missing:")
    for item in pre["missing"]:
        print(" -", item)
    print("Fix the missing items, then re-run. Do not rebuild the 420-case experiment.")
if pre["notes"]:
    print("Notes:", pre["notes"])



In [ ]:
# Launch the REAL Streamlit app. Does not rerun research stages.
import socket
import time
import urllib.request

def _port_free(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) != 0


def _missing_live_resources() -> list[str]:
    missing = []
    if not (V2_ROOT / "app" / "streamlit_app.py").is_file():
        missing.append("app/streamlit_app.py")
    if not (V2_ROOT / "knowledge_base" / "index" / "index_manifest.json").is_file():
        missing.append("knowledge_base/index (restore the existing Phase 6 index; do not rebuild)")
    try:
        from src.config import get_path, load_experiment_config
        from src.retrieval.preflight import validate_index_preflight
        validate_index_preflight(get_path(load_experiment_config(), "kb_index"))
    except Exception as exc:
        missing.append(f"Chroma index unusable: {exc}")
    return missing


if not RUN_LIVE_DEMO:
    print("RUN_LIVE_DEMO is False — not launching Streamlit.")
else:
    missing = _missing_live_resources()
    if missing:
        print("LIVE DEMO NOT STARTED")
        print("---------------------")
        print("Required resources are missing. Nothing was rebuilt.")
        for item in missing:
            print(" -", item)
        print()
        print("On Colab, restore MyDrive/MSc-RAG/artifacts/knowledge_base/ if the index is missing.")
        print("Do not run scripts/build_index.py, run_benchmark.py, run_calibration.py, or run_judge.py from this notebook.")
    else:
        port = 8501
        on_colab = Path("/content").exists()
        if not _port_free(port):
            print(f"Streamlit already appears to be running on port {port}.")
        else:
            cmd = [
                sys.executable, "-m", "streamlit", "run", "app/streamlit_app.py",
                "--server.port", str(port),
                "--server.headless", "true",
            ]
            if on_colab:
                cmd.extend(["--server.address", "0.0.0.0"])
            log_path = V2_ROOT / "results" / "logs" / "viva_streamlit.log"
            log_path.parent.mkdir(parents=True, exist_ok=True)
            log_f = open(log_path, "a", encoding="utf-8")
            env = os.environ.copy()
            env["PYTHONPATH"] = str(V2_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
            if on_colab:
                env["V2_LIVE_BACKEND"] = "llama_cpp"
                env["V2_FORBID_MOCK"] = "1"
            proc = subprocess.Popen(cmd, cwd=str(V2_ROOT), env=env, stdout=log_f, stderr=subprocess.STDOUT)
            print("Launched Streamlit pid", proc.pid)
            time.sleep(3)
        if on_colab:
            try:
                from google.colab.output import eval_js
                url = eval_js("google.colab.kernel.proxyPort(8501)")
                print("Open this Colab proxy URL (not 127.0.0.1 on the Mac):")
                print(" ", url)
            except Exception as exc:
                print("Colab proxy URL was not created:", exc)
                print("In Colab, use the kernel proxy for port 8501.")
        else:
            print("Open: http://127.0.0.1:8501")
            print("Pages: Live RAG Demo | Benchmark Results | Benchmark Questions")
            print("Official GPU Qwen answers need Colab Tesla T4 + llama.cpp + the existing index.")



## 17 Viva questions — code and implementation

Answers below are grounded in the final V2 files. If something is not in those files, that is stated.

### Dataset

**Q:** Where did your questions come from?  
**A:** FinQA inside T²-RAGBench (`dataset.huggingface_id: G4KMU/t2-ragbench`, subset FinQA). Frozen CSV: `data/final/selected_140_questions.csv`.  
**Where:** `config/experiment.yaml`, `src/data/select_140.py`

**Q:** Why 140, and why freeze them?  
**A:** `frozen_test_size: 140`, sampling seed 42, stratified with company/file caps. Freezing keeps the 420-case comparison stable.  
**Where:** `select_140.py` `stratified_sample`, `sampling_manifest.json`

**Q:** Why a DEV set of 40?  
**A:** Threshold selection must not use TEST. DEV CSV is `data/calibration/calibration_questions.csv`.  
**Where:** `src/calibration/data.py`, `threshold.lock.json`

### Document processing / chunking / embeddings / ChromaDB

**Q:** How are PDFs turned into evidence?  
**A:** Download page PDFs → PyMuPDF text → 900-character chunks with 150 overlap → BGE-small vectors → Chroma collection `finqa_source_pdfs`.  
**Where:** `pdf_fetch.py`, `extract.py`, `chunking.py`, `embeddings.py`, `index.py`

**Q:** Why top-k = 4 and cosine?  
**A:** `retrieval.top_k: 4` and Chroma `hnsw:space: cosine`. Similarity shown is `1 - distance`.  
**Where:** `experiment.yaml`, `retriever.py`

**Q:** Are gold answers in the index?  
**A:** Index manifest note: gold context fields are not ingested as retrieval documents.  
**Where:** `knowledge_base/index/index_manifest.json`

### Retrieval

**Q:** What does `retrieve()` do?  
**A:** Embed the question, query Chroma, return 4 `RetrievedChunk`s.  
**Where:** `src/retrieval/retriever.py`

**Q:** Why share retrieval?  
**A:** All three `run_*` functions call the same `retrieve(...)` with the same persist_dir, model, collection, and top_k.

### Single-Agent

**Q:** Why baseline?  
**A:** Retrieve + generate only; `verification_result=None`; `decision="ANSWER"`.  
**Where:** `single_agent.py`

**Q:** Why always answer?  
**A:** No abstention function is called; decision is hard-coded ANSWER.

### Multi-Agent / verification

**Q:** Does the checker rewrite the answer?  
**A:** No. `run_multi_agent` keeps `answer` from generation and only attaches `verification_result`.  
**Where:** `multi_agent.py`, `verification.py`

**Q:** How is verification scored?  
**A:** Mean of token overlap and an LLM 0–1 support score; ≥ 0.50 is VERIFIED.

**Q:** LangGraph?  
**A:** **Not in the final V2 implementation.** There is no `langgraph` import. Flow is sequential Python.

### UQ / threshold

**Q:** Confidence formula?  
**A:** `confidence = average([retrieval_score, verification_score])` with method `mean_retrieval_verification`.  
**Where:** `uncertainty.py`

**Q:** Why 0.65?  
**A:** DEV sweep, coverage floor 0.50, maximise selective accuracy, lowest-T tie-break. Locked; not TEST.  
**Where:** `calibration/select.py`, `threshold.lock.json`

**Q:** Is confidence a probability?  
**A:** No. Operational decision score. Formal ECE/Brier were not implemented.

### Prompts / Qwen / llama.cpp

**Q:** What model?  
**A:** Qwen3-8B, GGUF `Q4_K_M`, `llama.cpp`, n_ctx 4096. Official remote GPU: Tesla T4.  
**Where:** `experiment.yaml`, `llama_cpp_backend.py`

**Q:** Why llama.cpp?  
**A:** It is the primary Colab GPU backend for the GGUF file. `create_backend` prefers `llama_cpp` when that extra is installed.

**Q:** Ollama?  
**A:** Optional local-dev backend only. Live demo runtime guard forbids it when locked.

### Evaluation / judge / statistics / errors

**Q:** Correctness metric?  
**A:** `numeric_match` vs `program_answer`.  
**Where:** `evaluation/numeric.py`, `metrics.score_case`

**Q:** Judge sees gold?  
**A:** No. `judge.py` docstring: no gold context/answer in the prompt; leakage check `prompt_contains_forbidden`.

**Q:** Official RAGAS?  
**A:** No. Label is custom / RAGAS-inspired.

**Q:** p = 0.6776?  
**A:** Exact McNemar SA vs MA displayed correctness; not significant.  
**Where:** `statistics/tests.py`, `results/metrics/phase17_tests.csv`

**Q:** Error types?  
**A:** Rule-based taxonomy including retrieval_failure vs incorrect_numerical_reasoning vs unsupported_claim vs abstention.  
**Where:** `error_analysis/taxonomy.py`, `phase18_error_summary.csv`

### Streamlit / testing / reproducibility / limitations

**Q:** How do I run the artefact?  
**A:** `PYTHONPATH=. streamlit run app/streamlit_app.py` from `V2/`, or this notebook’s live-demo cell. GPU answers: Phase 21 Colab notebook.

**Q:** How is it tested?  
**A:** `V2/tests/` (pytest) plus recorded live-artefact checks. This notebook does not rerun those suites.

**Q:** How is it reproducible?  
**A:** Frozen CSVs, SHA-256 pins in `statistics/constants.py`, locked T, saved Phase 15–18 artefacts, config YAML.

**Q:** Main limitations in code/design?  
**A:** One retrieval setup; 140-question FinQA sample; same Qwen3-8B for generate/check/judge; confidence not calibrated; no user study.

**Q:** Would removing Chroma break the project?  
**A:** Yes. Retrieval and therefore all three architectures depend on that index.



---

**End of walkthrough.** Run the configuration cell and the helper cell first. Then jump to the section the examiner asks about. For the demo, run **16 Live demo**.

Do not commit this notebook.

